# 🚀 Google Colab Pro Scheduled Runner
### Autonomous In-The-Wild Crawling, 5-Category Vietnamese Benchmark & Continual Fine-Tuning

This notebook is specifically configured for **Google Colab Pro Scheduled Runs** (Headless execution without human presence).

**What this run performs**:
1. **Auto-Mounts** Google Drive (`/content/drive/MyDrive/Thread_Data/`).
2. **Pulls** the latest code from `ThangCao106/DeepFake-Detection-Agent` (`main` branch).
3. **Crawls** authentic in-the-wild Meta Threads media with Vietnamese targeting (`Accept-Language: vi-VN`).
4. **Generates** 5-Category Vietnamese Facial Benchmark datasets (`vn_id_official`, `vn_selfie_lifestyle`, `vn_street_outdoor`, `vn_occlusion_glasses_mask`, `vn_mature_aging`).
5. **Extracts** 2D Haar Wavelet pore energy ($E_{high}$) and 2D FFT spectral anomaly peaks ($Z_{peak}$).
6. **Mines** boundary hard negative samples ($H(P) \ge 0.70$ and classifier errors).
7. **Fine-tunes** the ViT backbone with **Focal Loss** ($\gamma=2.0, \alpha=0.25$).
8. **Persists** updated checkpoints (`model_continual_latest.pt`) directly to Google Drive.

--- 
## Step 1: Auto-Mount Google Drive

In [ ]:
import os, sys, time
from google.colab import drive

# Auto-mounts Google Drive without manual popup during scheduled runs
drive.mount('/content/drive', force_remount=False)
print('✅ Google Drive mounted successfully at /content/drive/MyDrive')

--- 
## Step 2: Clone/Pull Repository & Install Dependencies

In [ ]:
# Pull latest code from GitHub
!git clone https://github.com/ThangCao106/DeepFake-Detection-Agent.git /content/DeepFake-Detection-Agent || (cd /content/DeepFake-Detection-Agent && git fetch origin main && git reset --hard origin/main && git pull origin main)
%cd /content/DeepFake-Detection-Agent

# Install dependencies in quiet mode
!pip install -q -r requirements.txt transformers timm
print('✅ Dependencies and repository ready.')

--- 
## Step 3: Run Batch Ingestion, Vietnamese Taxonomy Generation & Focal Loss Fine-Tuning

In [ ]:
# Execute a single comprehensive batch
# (Do NOT use --loop in Scheduled Runs so the notebook cleanly exits after completion to save compute units)
!python colab_sync_to_gdrive.py \
    --samples 25 \
    --images 100 \
    --videos 50 \
    --audios 50 \
    --train \
    --epochs 3


--- 
## Step 4: Verification & Execution Summary

In [ ]:
from pathlib import Path
date_str = time.strftime('%Y-%m-%d')
gdrive_folder = Path('/content/drive/MyDrive/Thread_Data') / date_str
checkpoint_file = Path('/content/drive/MyDrive/Thread_Data/checkpoints/model_continual_latest.pt')
manifest_file = gdrive_folder / 'dataset_manifest.jsonl'
summary_file = gdrive_folder / 'dataset_summary.md'

print('\n' + '='*65)
print('🎉 GOOGLE COLAB PRO SCHEDULED RUN COMPLETED SUCCESSFULLY!')
print('='*65)
print(f'📅 Run Date: {date_str}')
print(f'📁 Google Drive Folder: {gdrive_folder}')
print(f'   - Manifest Exists: {manifest_file.exists()}')
print(f'   - Summary Exists: {summary_file.exists()}')
print(f'💾 Model Checkpoint: {checkpoint_file}')
print(f'   - Checkpoint Exists: {checkpoint_file.exists()}')
print('='*65)
print('Colab VM will automatically shut down after completion to conserve compute units.')
